# PoC 3: Video Behavior Analysis

**Question:** Can we extract useful behavioral metrics from companion `_v1.mp4` video files?

Each `.mat` neurophysiology chunk may have a companion video recorded simultaneously.
If motion/behavior correlates with neural features, we can flag epochs where behavior
confounds evoked responses (e.g., animal moving during stimulation).

**Approach:**
1. Survey which sessions have video available
2. Visual inspection of frame quality
3. Frame-differencing motion energy (no ML required)
4. Freeze detection
5. Batch analysis across sessions
6. Correlate motion with neural features
7. (Optional) Pose estimation with pretrained model

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('.')))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2

from _common import load_db, video_path_for_mat, load_video_frames, load_evoked_feature_matrix

plt.style.use('dark_background')
DB_PATH = os.path.join('..', 'data', 'monitor.db')

## 1. Survey available videos

In [ ]:
conn = load_db(DB_PATH)

# Count total files vs files with video
total_files = pd.read_sql_query("SELECT COUNT(*) AS n FROM processed_files", conn).iloc[0]['n']
video_files = pd.read_sql_query(
    "SELECT COUNT(*) AS n FROM processed_files WHERE has_video = 1", conn
).iloc[0]['n']

print(f"Total processed files: {total_files}")
print(f"Files with companion video: {video_files} ({100*video_files/max(total_files,1):.1f}%)")
print()

# List unique sessions that have video
video_df = pd.read_sql_query(
    """SELECT id, file_path, session_dir, session_name, chunk_datetime
       FROM processed_files
       WHERE has_video = 1
       ORDER BY chunk_datetime""",
    conn,
)
sessions_with_video = video_df['session_dir'].nunique()
print(f"Unique sessions with video: {sessions_with_video}")
print(video_df[['session_name', 'chunk_datetime']].head(10).to_string(index=False))

# Probe metadata for one sample video
sample_row = video_df.iloc[0]
sample_video = video_path_for_mat(sample_row['file_path'])

if sample_video is not None:
    cap = cv2.VideoCapture(sample_video)
    assert cap.isOpened(), f"Cannot open {sample_video}"

    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fourcc_int = int(cap.get(cv2.CAP_PROP_FOURCC))
    codec = "".join([chr((fourcc_int >> (8 * i)) & 0xFF) for i in range(4)])
    duration_sec = frame_count / fps if fps > 0 else 0
    cap.release()

    print(f"\nSample video: {os.path.basename(sample_video)}")
    print(f"  Resolution : {width} x {height}")
    print(f"  FPS        : {fps:.2f}")
    print(f"  Codec      : {codec}")
    print(f"  Frames     : {frame_count}")
    print(f"  Duration   : {duration_sec:.1f} sec ({duration_sec/60:.1f} min)")
else:
    print("\nWARNING: Sample video file not accessible (network share may be down).")
    print("Subsequent cells that require video access will be skipped.")

conn.close()

## 2. Frame exploration

In [ ]:
# Pick one video and load 9 evenly-spaced frames
if sample_video is not None:
    cap = cv2.VideoCapture(sample_video)
    assert cap.isOpened(), f"Cannot open {sample_video}"

    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    native_fps = cap.get(cv2.CAP_PROP_FPS)
    cap.release()

    duration = total / native_fps
    n_samples = 9
    # Pre-allocate sample times evenly across the video
    sample_times = np.linspace(0, duration * 0.95, n_samples)

    frames_9 = []
    for t in sample_times:
        loaded, _, _ = load_video_frames(sample_video, start_sec=t, end_sec=t + 0.5, max_frames=1)
        if len(loaded) > 0:
            frames_9.append(loaded[0])

    assert len(frames_9) > 0, "No frames loaded from sample video"

    # Display as 3x3 grid
    fig, axes = plt.subplots(3, 3, figsize=(14, 10))
    fig.suptitle(f"Sample frames: {os.path.basename(sample_video)}", fontsize=14)
    for idx in range(9):
        ax = axes[idx // 3][idx % 3]
        if idx < len(frames_9):
            # cv2 loads BGR; convert to RGB for matplotlib
            rgb = cv2.cvtColor(frames_9[idx], cv2.COLOR_BGR2RGB)
            ax.imshow(rgb)
            ax.set_title(f"t = {sample_times[idx]:.1f}s", fontsize=10)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

    # Grayscale histogram of one frame
    gray_frame = cv2.cvtColor(frames_9[0], cv2.COLOR_BGR2GRAY)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.imshow(gray_frame, cmap='gray')
    ax1.set_title('Grayscale frame')
    ax1.axis('off')

    ax2.hist(gray_frame.ravel(), bins=256, range=(0, 256), color='cyan', alpha=0.7)
    ax2.set_xlabel('Pixel intensity')
    ax2.set_ylabel('Count')
    ax2.set_title('Grayscale histogram')
    plt.tight_layout()
    plt.show()

    print(f"Frame shape: {frames_9[0].shape}")
    print(f"Grayscale mean: {gray_frame.mean():.1f}, std: {gray_frame.std():.1f}")
else:
    print("Skipped: no video accessible.")

## 3. Motion energy (no model needed)

In [ ]:
def compute_motion_energy(video_path, sample_every_n=3, resize_width=320, max_frames=50000):
    """Frame-differencing motion energy -- no ML required.

    Returns (time_sec_array, motion_energy_array, native_fps).
    """
    assert os.path.isfile(video_path), f"Cannot find {video_path}"
    assert sample_every_n >= 1, "sample_every_n must be >= 1"

    cap = cv2.VideoCapture(video_path)
    assert cap.isOpened(), f"Cannot open {video_path}"

    fps = cap.get(cv2.CAP_PROP_FPS)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    assert fps > 0, f"Invalid FPS: {fps}"

    # Pre-allocate arrays with upper bound size
    max_possible = min(total // sample_every_n, max_frames)
    times = np.zeros(max_possible, dtype=np.float64)
    energies = np.zeros(max_possible, dtype=np.float64)
    count = 0

    prev_gray = None
    frame_idx = 0
    n_read = 0
    loop_limit = min(total, max_frames * sample_every_n)

    for _ in range(loop_limit):
        ret, frame = cap.read()
        if not ret:
            break
        if frame_idx % sample_every_n == 0:
            h, w = frame.shape[:2]
            scale = resize_width / w
            new_h = int(h * scale)
            small = cv2.resize(frame, (resize_width, new_h))
            gray = cv2.cvtColor(small, cv2.COLOR_BGR2GRAY).astype(np.float32)

            if prev_gray is not None:
                diff = np.mean(np.abs(gray - prev_gray))
                times[count] = frame_idx / fps
                energies[count] = diff
                count += 1
            prev_gray = gray
            n_read += 1
            if n_read >= max_frames:
                break
        frame_idx += 1

    cap.release()

    # Trim to actual size
    return times[:count], energies[:count], fps


print("compute_motion_energy() defined.")

In [ ]:
# Run on sample video and plot
if sample_video is not None:
    times_me, energies_me, vid_fps = compute_motion_energy(sample_video)
    assert len(times_me) > 0, "No motion energy computed"

    me_mean = np.mean(energies_me)
    me_std = np.std(energies_me)
    me_max = np.max(energies_me)

    freeze_thresh = me_mean - 1.0 * me_std
    high_thresh = me_mean + 2.0 * me_std
    pct_frozen = 100.0 * np.sum(energies_me < freeze_thresh) / len(energies_me)

    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(times_me / 60.0, energies_me, linewidth=0.5, color='cyan', alpha=0.8)
    ax.axhline(freeze_thresh, color='blue', linestyle='--', alpha=0.7, label=f'Freeze threshold ({freeze_thresh:.1f})')
    ax.axhline(high_thresh, color='red', linestyle='--', alpha=0.7, label=f'High motion ({high_thresh:.1f})')
    ax.axhline(me_mean, color='white', linestyle=':', alpha=0.5, label=f'Mean ({me_mean:.1f})')
    ax.set_xlabel('Time (min)')
    ax.set_ylabel('Motion energy (mean abs diff)')
    ax.set_title(f'Motion energy: {os.path.basename(sample_video)}')
    ax.legend(loc='upper right', fontsize=9)
    plt.tight_layout()
    plt.show()

    print(f"Video FPS: {vid_fps:.1f}")
    print(f"Motion energy frames: {len(energies_me)}")
    print(f"Mean: {me_mean:.2f}, Std: {me_std:.2f}, Max: {me_max:.2f}")
    print(f"% time below freeze threshold: {pct_frozen:.1f}%")
else:
    print("Skipped: no video accessible.")

## 4. Freeze detection

In [ ]:
def detect_freezing(times, energies, threshold, min_duration_sec=2.0):
    """Detect freeze epochs where motion energy < threshold for >= min_duration.

    Returns:
        freeze_pct: float, percentage of total time in freeze state
        freeze_episodes: int, number of freeze episodes
        freeze_intervals: list of (start_sec, end_sec) tuples
    """
    assert len(times) == len(energies), "times and energies must have same length"
    assert len(times) > 1, "Need at least 2 data points"

    is_frozen = energies < threshold
    n = len(times)

    # Pre-allocate for freeze intervals (upper bound: n/2 episodes)
    max_episodes = n // 2 + 1
    interval_starts = np.zeros(max_episodes, dtype=np.float64)
    interval_ends = np.zeros(max_episodes, dtype=np.float64)
    episode_count = 0

    # Walk through frozen/not-frozen transitions
    in_freeze = False
    ep_start = 0.0

    for i in range(n):
        if is_frozen[i] and not in_freeze:
            in_freeze = True
            ep_start = times[i]
        elif not is_frozen[i] and in_freeze:
            in_freeze = False
            ep_end = times[i]
            if (ep_end - ep_start) >= min_duration_sec:
                interval_starts[episode_count] = ep_start
                interval_ends[episode_count] = ep_end
                episode_count += 1

    # Close any open freeze epoch at the end
    if in_freeze:
        ep_end = times[-1]
        if (ep_end - ep_start) >= min_duration_sec:
            interval_starts[episode_count] = ep_start
            interval_ends[episode_count] = ep_end
            episode_count += 1

    # Trim to actual size
    interval_starts = interval_starts[:episode_count]
    interval_ends = interval_ends[:episode_count]

    # Build return list
    freeze_intervals = [(interval_starts[i], interval_ends[i]) for i in range(episode_count)]

    total_freeze_time = np.sum(interval_ends - interval_starts)
    total_time = times[-1] - times[0] if len(times) > 1 else 1.0
    freeze_pct = 100.0 * total_freeze_time / max(total_time, 1e-9)

    return freeze_pct, episode_count, freeze_intervals


# Run on sample video
if sample_video is not None:
    freeze_pct, n_episodes, intervals = detect_freezing(
        times_me, energies_me, threshold=freeze_thresh, min_duration_sec=2.0
    )

    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(times_me / 60.0, energies_me, linewidth=0.5, color='cyan', alpha=0.8)
    ax.axhline(freeze_thresh, color='blue', linestyle='--', alpha=0.5)

    for start, end in intervals:
        ax.axvspan(start / 60.0, end / 60.0, alpha=0.25, color='blue', label=None)

    ax.set_xlabel('Time (min)')
    ax.set_ylabel('Motion energy')
    ax.set_title(f'Freeze detection: {n_episodes} episodes, {freeze_pct:.1f}% frozen')
    plt.tight_layout()
    plt.show()

    print(f"Freeze threshold: {freeze_thresh:.2f}")
    print(f"Freeze episodes: {n_episodes}")
    print(f"Freeze %: {freeze_pct:.1f}%")
    if n_episodes > 0:
        durations = [end - start for start, end in intervals]
        print(f"Mean episode duration: {np.mean(durations):.1f}s")
        print(f"Max episode duration: {np.max(durations):.1f}s")
else:
    print("Skipped: no video accessible.")

## 5. Batch analysis across sessions

In [ ]:
# Run motion energy on all available videos
conn = load_db(DB_PATH)
all_video_df = pd.read_sql_query(
    """SELECT id, file_path, session_dir, session_name, chunk_datetime
       FROM processed_files
       WHERE has_video = 1
       ORDER BY chunk_datetime""",
    conn,
)
conn.close()

n_total = len(all_video_df)
print(f"Videos to process: {n_total}")

# Pre-allocate results columns
result_file_ids = np.zeros(n_total, dtype=np.int64)
result_mean_motion = np.full(n_total, np.nan)
result_std_motion = np.full(n_total, np.nan)
result_max_motion = np.full(n_total, np.nan)
result_freeze_pct = np.full(n_total, np.nan)
result_freeze_eps = np.zeros(n_total, dtype=np.int64)
result_session_dirs = [None] * n_total
result_datetimes = [None] * n_total
result_success = np.zeros(n_total, dtype=bool)

n_processed = 0
n_failed = 0

for idx in range(n_total):
    row = all_video_df.iloc[idx]
    fid = row['id']
    fpath = row['file_path']

    result_file_ids[idx] = fid
    result_session_dirs[idx] = row['session_dir']
    result_datetimes[idx] = row['chunk_datetime']

    try:
        vpath = video_path_for_mat(fpath)
        if vpath is None:
            n_failed += 1
            continue

        t_arr, e_arr, _ = compute_motion_energy(vpath, sample_every_n=5)
        if len(e_arr) < 10:
            n_failed += 1
            continue

        m_mean = np.mean(e_arr)
        m_std = np.std(e_arr)
        m_max = np.max(e_arr)
        f_thresh = m_mean - 1.0 * m_std
        f_pct, f_eps, _ = detect_freezing(t_arr, e_arr, threshold=f_thresh)

        result_mean_motion[idx] = m_mean
        result_std_motion[idx] = m_std
        result_max_motion[idx] = m_max
        result_freeze_pct[idx] = f_pct
        result_freeze_eps[idx] = f_eps
        result_success[idx] = True
        n_processed += 1

    except Exception as exc:
        n_failed += 1
        if idx < 3:
            print(f"  Error on file {fid}: {exc}")

    # Progress
    if (idx + 1) % 50 == 0 or idx == n_total - 1:
        print(f"  [{idx+1}/{n_total}] processed={n_processed}, failed={n_failed}")

# Build results DataFrame
batch_df = pd.DataFrame({
    'file_id': result_file_ids,
    'session_dir': result_session_dirs,
    'chunk_datetime': result_datetimes,
    'mean_motion': result_mean_motion,
    'std_motion': result_std_motion,
    'max_motion': result_max_motion,
    'freeze_pct': result_freeze_pct,
    'freeze_episodes': result_freeze_eps,
})
batch_df = batch_df[result_success].copy()
batch_df['chunk_datetime'] = pd.to_datetime(batch_df['chunk_datetime'])

print(f"\nDone. {n_processed} videos processed, {n_failed} failed.")
print(batch_df.describe())

In [ ]:
# Visualize batch results
if len(batch_df) > 0:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # (a) Motion energy over time per session
    ax = axes[0, 0]
    sessions = batch_df['session_dir'].unique()
    max_sessions_plot = 10  # fixed bound for legend readability
    for i, sess in enumerate(sessions[:max_sessions_plot]):
        subset = batch_df[batch_df['session_dir'] == sess].sort_values('chunk_datetime')
        label = os.path.basename(str(sess))[:20]
        ax.plot(subset['chunk_datetime'], subset['mean_motion'], marker='o', markersize=3, label=label)
    ax.set_xlabel('Time')
    ax.set_ylabel('Mean motion energy')
    ax.set_title('Motion energy over time by session')
    ax.legend(fontsize=7, loc='upper right')
    ax.tick_params(axis='x', rotation=45)

    # (b) Freeze % distribution
    ax = axes[0, 1]
    ax.hist(batch_df['freeze_pct'].dropna(), bins=30, color='steelblue', edgecolor='white')
    ax.set_xlabel('Freeze %')
    ax.set_ylabel('Count')
    ax.set_title('Freeze % distribution across videos')

    # (c) Mean motion distribution
    ax = axes[1, 0]
    ax.hist(batch_df['mean_motion'].dropna(), bins=30, color='teal', edgecolor='white')
    ax.set_xlabel('Mean motion energy')
    ax.set_ylabel('Count')
    ax.set_title('Mean motion energy distribution')

    # (d) Heatmap: session vs hour-of-day, color = mean motion
    ax = axes[1, 1]
    heatmap_df = batch_df.copy()
    heatmap_df['hour'] = heatmap_df['chunk_datetime'].dt.hour
    heatmap_df['session_short'] = heatmap_df['session_dir'].apply(lambda x: os.path.basename(str(x))[:15])
    pivot = heatmap_df.pivot_table(values='mean_motion', index='session_short', columns='hour', aggfunc='mean')
    if pivot.shape[0] > 0 and pivot.shape[1] > 0:
        sns.heatmap(pivot, ax=ax, cmap='viridis', cbar_kws={'label': 'Mean motion'})
        ax.set_title('Session vs hour-of-day')
    else:
        ax.text(0.5, 0.5, 'Insufficient data for heatmap', ha='center', va='center', transform=ax.transAxes)
        ax.set_title('Session vs hour-of-day')

    plt.tight_layout()
    plt.show()
else:
    print("No batch results to visualize.")

## 6. Correlate motion with neural features

In [ ]:
from scipy import stats

# Load evoked features
evoked_df = load_evoked_feature_matrix(DB_PATH)
print(f"Evoked features: {len(evoked_df)} epochs from {evoked_df['file_id'].nunique()} files")

# Aggregate neural features per file_id (mean per file)
neural_cols = [
    'peak_amplitude', 'trough_amplitude', 'peak_to_trough', 'rms_amplitude',
    'peak_latency_ms', 'trough_latency_ms', 'max_slope', 'line_length',
    'log_auc', 'early_area', 'late_area', 'early_late_ratio',
    'template_correlation', 'pca_recon_error', 'variance',
    'autocorrelation', 'recovery_tau', 'recovery_slope',
]
neural_agg = evoked_df.groupby('file_id')[neural_cols].mean().reset_index()

# Join with motion energy results
merged = batch_df.merge(neural_agg, on='file_id', how='inner')
print(f"Merged records (files with both video + neural features): {len(merged)}")

if len(merged) >= 5:
    # Scatter plots: motion vs key neural features
    key_neural = ['peak_amplitude', 'rms_amplitude', 'template_correlation']
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    for i, col in enumerate(key_neural):
        ax = axes[i]
        valid = merged[['mean_motion', col]].dropna()
        ax.scatter(valid['mean_motion'], valid[col], alpha=0.6, s=20, color='cyan')
        if len(valid) >= 3:
            r_pearson, p_pearson = stats.pearsonr(valid['mean_motion'], valid[col])
            r_spearman, p_spearman = stats.spearmanr(valid['mean_motion'], valid[col])
            ax.set_title(f'{col}\nr={r_pearson:.3f} (p={p_pearson:.3f})', fontsize=10)
        ax.set_xlabel('Mean motion energy')
        ax.set_ylabel(col)

    plt.tight_layout()
    plt.show()

    # Full correlation matrix: motion metrics vs neural features
    motion_cols = ['mean_motion', 'std_motion', 'max_motion', 'freeze_pct']
    corr_cols = motion_cols + neural_cols
    available_cols = [c for c in corr_cols if c in merged.columns]
    corr_matrix = merged[available_cols].corr(method='spearman')

    # Extract the motion-vs-neural block
    motion_available = [c for c in motion_cols if c in corr_matrix.columns]
    neural_available = [c for c in neural_cols if c in corr_matrix.columns]
    cross_corr = corr_matrix.loc[neural_available, motion_available]

    fig, ax = plt.subplots(figsize=(8, 10))
    sns.heatmap(
        cross_corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
        vmin=-1, vmax=1, ax=ax, cbar_kws={'label': 'Spearman r'}
    )
    ax.set_title('Neural features vs Motion metrics (Spearman)')
    plt.tight_layout()
    plt.show()

    # Find strongest correlation
    abs_corr = cross_corr.abs()
    max_idx = abs_corr.stack().idxmax()
    best_neural_feat = max_idx[0]
    best_motion_feat = max_idx[1]
    best_correlation = cross_corr.loc[best_neural_feat, best_motion_feat]
    print(f"\nStrongest correlation: {best_neural_feat} vs {best_motion_feat}")
    print(f"  Spearman r = {best_correlation:.3f}")
else:
    print("Not enough merged data for correlation analysis.")
    best_correlation = 0.0

## 7. Pose estimation exploration (optional)

In [ ]:
# This cell is optional -- only run if DLC or SLEAP is installed
DLC_AVAILABLE = False

try:
    from dlclive import DLCLive, Processor
    DLC_AVAILABLE = True
    print("DeepLabCut-Live available")
except ImportError:
    DLC_AVAILABLE = False
    print("DLC not installed -- skipping pose estimation")
    print("Install with: pip install deeplabcut-live")

if DLC_AVAILABLE and sample_video is not None:
    # Try the superanimal_topviewmouse pretrained model on a few frames
    # to evaluate if keypoints are reasonable for this camera setup
    print("\nTesting DLC on sample frames...")

    # Load a few frames for testing
    test_frames, _, _ = load_video_frames(sample_video, start_sec=30, end_sec=35, max_frames=5)
    assert len(test_frames) > 0, "No test frames loaded"

    try:
        dlc_proc = Processor()
        # Use the superanimal pretrained model
        dlc_live = DLCLive("superanimal_topviewmouse", processor=dlc_proc)

        # Init on first frame
        first_pose = dlc_live.init_inference(test_frames[0])
        print(f"Keypoints shape: {first_pose.shape}")
        print(f"Keypoint names: {dlc_live.cfg.get('all_joints_names', 'N/A')}")

        # Run on remaining frames
        n_test = min(len(test_frames), 5)
        fig, axes = plt.subplots(1, n_test, figsize=(4 * n_test, 4))
        if n_test == 1:
            axes = [axes]

        for i in range(n_test):
            if i == 0:
                pose = first_pose
            else:
                pose = dlc_live.get_pose(test_frames[i])

            rgb = cv2.cvtColor(test_frames[i], cv2.COLOR_BGR2RGB)
            ax = axes[i]
            ax.imshow(rgb)

            # Plot keypoints with confidence > 0.5
            confident = pose[:, 2] > 0.5
            ax.scatter(
                pose[confident, 0], pose[confident, 1],
                c='lime', s=30, marker='o', edgecolors='white', linewidths=0.5
            )
            ax.set_title(f"Frame {i}: {np.sum(confident)}/{len(pose)} kpts", fontsize=9)
            ax.axis('off')

        plt.suptitle('DLC pose estimation (superanimal_topviewmouse)', fontsize=12)
        plt.tight_layout()
        plt.show()

        mean_conf = np.mean(first_pose[:, 2])
        print(f"Mean keypoint confidence: {mean_conf:.3f}")

    except Exception as exc:
        print(f"DLC inference failed: {exc}")
        print("Model may need to be downloaded first.")

elif not DLC_AVAILABLE:
    print("\nTo explore pose estimation later, install DLC:")
    print("  pip install deeplabcut-live")
else:
    print("No video accessible for pose estimation test.")

## 8. Verdict

In [ ]:
# Evaluate two criteria:
# (a) Does motion energy correlate with evoked feature variability?
# (b) Can a pretrained pose model give clean keypoints?

motion_neural_corr = abs(best_correlation)

if motion_neural_corr > 0.3:
    verdict = "SHIP"
    reason = (
        f"Motion energy correlates with neural features (r={best_correlation:.3f}). "
        f"Cheap win -- no ML model needed."
    )
elif motion_neural_corr > 0.15:
    verdict = "ITERATE"
    reason = (
        f"Weak correlation (r={best_correlation:.3f}). "
        f"May improve with pose-based features."
    )
else:
    verdict = "DROP"
    reason = (
        f"No meaningful correlation (r={best_correlation:.3f}). "
        f"Video behavior doesn't predict neural state."
    )

print(f"VERDICT: {verdict}")
print(f"Reason: {reason}")
if DLC_AVAILABLE:
    print("Note: Pose estimation was tested -- see results above.")
else:
    print("Note: Pose estimation not tested (DLC not installed).")